# VISTA smoke/normal test

**Reference:** Huuki-Myers snRNA-seq counts + metadata  
**Target:** Br8667 Xenium AnnData  

This notebook runs **one VISTA run**, not 10 repeated runs.  
The Huuki-Myers reference itself may contain 10 biological samples/donors in metadata.

Use `RUN_MODE = "smoke"` for testing GPU feasibility.  
Use `RUN_MODE = "full"` only after the smoke run works.


In [ ]:
from pathlib import Path
import sys
import time
import random
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
from scipy import sparse
import torch

PROJECT_ROOT = Path("/users/mjabin/projects/GeneBridge")

# Correct reference: Huuki-Myers snRNA-seq
HUUKI_METADATA_CSV = PROJECT_ROOT / "outputs/huuki_myers/tables/huuki_snrna_metadata.csv"
HUUKI_COUNTS_CSV = PROJECT_ROOT / "outputs/huuki_myers/tables/huuki_snrna_brain_counts.csv"

# Correct target: Br8667 Xenium
XENIUM_TARGET_H5AD = PROJECT_ROOT / "data/processed/imputation_beta/Br8667/spatial_data_xenium_Br8667_vista.h5ad"

OUT_DIR = PROJECT_ROOT / "outputs/imputation_beta/Br8667/smoke_tests/vista_huuki_snrna_reference_xenium_target"
OUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_MODE = "smoke"   # "smoke" or "full"

# Smoke mode = one small feasibility run, not 10 repeated runs
SMOKE_REF_CELLS = 2000
SMOKE_TARGET_CELLS = 5000
SMOKE_GENES = 100
SMOKE_EPOCHS = 2

# Full mode = all available cells/common genes. Use only after smoke works.
FULL_EPOCHS = 200

BATCH_SIZE = 128
N_LATENT = 16
NEIGHBOR_SIZE = 20
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Project root:", PROJECT_ROOT)
print("Reference metadata:", HUUKI_METADATA_CSV)
print("Reference counts:", HUUKI_COUNTS_CSV)
print("Target Xenium:", XENIUM_TARGET_H5AD)
print("Run mode:", RUN_MODE)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Project root: /users/mjabin/projects/GeneBridge
Reference metadata: /users/mjabin/projects/GeneBridge/outputs/huuki_myers/tables/huuki_snrna_metadata.csv
Reference counts: /users/mjabin/projects/GeneBridge/outputs/huuki_myers/tables/huuki_snrna_brain_counts.csv
Target Xenium: /users/mjabin/projects/GeneBridge/data/processed/imputation_beta/Br8667/spatial_data_xenium_Br8667_vista.h5ad
Run mode: smoke
CUDA available: False


In [3]:
# Optional safety patch for local VISTA when validation_size=0.0
# This avoids val_dataloader=None crash in some local VISTA versions.

vista_model_path = PROJECT_ROOT / "src/imputation/VISTA/vista/_model.py"

if vista_model_path.exists():
    text = vista_model_path.read_text()
    old = '''val = ds.val_dataloader()
            val_dls.append(val)
            val.mode = i'''
    new = '''val = ds.val_dataloader()
            if val is not None:
                val_dls.append(val)
                val.mode = i'''
    if old in text and new not in text:
        vista_model_path.write_text(text.replace(old, new))
        print("Patched VISTA _model.py for validation_size=0.0")
    else:
        print("VISTA patch not needed or already applied.")
else:
    print("VISTA _model.py not found at:", vista_model_path)


VISTA patch not needed or already applied.


In [4]:
# Import local VISTA

VISTA_ROOT = PROJECT_ROOT / "src/imputation/VISTA"
sys.path.insert(0, str(VISTA_ROOT))

from vista import GIMVI_GCN

print("Imported GIMVI_GCN from local VISTA.")


/scratch/mjabin/conda_envs/Imputation/lib/python3.9/site-packages/lightning_fabric/__init__.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)
Global seed set to 0


Imported GIMVI_GCN from local VISTA.


In [5]:
# Load Huuki-Myers snRNA metadata and count matrix

print("Loading Huuki-Myers metadata...")
meta = pd.read_csv(HUUKI_METADATA_CSV, index_col=0)

print("Loading Huuki-Myers counts...")
counts = pd.read_csv(HUUKI_COUNTS_CSV, index_col=0)

meta.index = meta.index.astype(str)
counts.index = counts.index.astype(str)
counts.columns = counts.columns.astype(str)

print("metadata shape:", meta.shape)
print("counts shape:", counts.shape)
print("metadata columns:")
print(list(meta.columns))

# Detect count matrix orientation.
# VISTA/AnnData needs cells x genes.
row_matches = len(set(counts.index) & set(meta.index))
col_matches = len(set(counts.columns) & set(meta.index))

print("count row IDs matching metadata cell IDs:", row_matches)
print("count column IDs matching metadata cell IDs:", col_matches)

if col_matches > row_matches:
    print("Detected counts orientation: genes x cells. Transposing to cells x genes.")
    counts = counts.T
elif row_matches > 0:
    print("Detected counts orientation: cells x genes.")
else:
    raise ValueError(
        "Could not match count matrix rows or columns with metadata cell IDs. "
        "Check whether metadata index and count matrix cell IDs match."
    )

common_cells = counts.index.intersection(meta.index)
print("common cells between counts and metadata:", len(common_cells))

counts = counts.loc[common_cells]
meta = meta.loc[common_cells]

# Build sparse AnnData reference
adata_ref = ad.AnnData(
    X=sparse.csr_matrix(counts.values),
    obs=meta.copy(),
    var=pd.DataFrame(index=counts.columns.astype(str)),
)

adata_ref.obs_names = adata_ref.obs_names.astype(str)
adata_ref.var_names = adata_ref.var_names.astype(str)

print("Huuki-Myers snRNA reference AnnData:")
print(adata_ref)

# Show likely sample/donor columns if present
candidate_sample_cols = [
    "sample", "sample_id", "Sample", "SampleID",
    "donor", "donor_id", "Donor", "subject", "subject_id",
    "BrNum", "brnum", "brain", "brain_id"
]

for col in candidate_sample_cols:
    if col in adata_ref.obs.columns:
        vals = adata_ref.obs[col].astype(str).unique()
        print(f"Possible sample column: {col} | n={len(vals)}")
        print(vals[:20])


Loading Huuki-Myers metadata...
Loading Huuki-Myers counts...
metadata shape: (77604, 34)
counts shape: (10, 1)
metadata columns:
['Barcode', 'key', 'SAMPLE_ID', 'pos', 'BrNum', 'round', 'Position', 'age', 'sex', 'diagnosis', 'sum', 'detected', 'subsets_Mito_sum', 'subsets_Mito_detected', 'subsets_Mito_percent', 'total', 'high_mito', 'low_sum', 'low_detected', 'discard_auto', 'doubletScore', 'prelimCluster', 'collapsedCluster', 'kmeans', 'sizeFactor', 'cellType_broad_k', 'cellType_k', 'cellType_broad_hc', 'cellType_hc', 'cellType_layer', 'layer_annotation', 'cell_id', 'UMAP1', 'UMAP2']
count row IDs matching metadata cell IDs: 0
count column IDs matching metadata cell IDs: 0


ValueError: Could not match count matrix rows or columns with metadata cell IDs. Check whether metadata index and count matrix cell IDs match.

In [ ]:
# Load Br8667 Xenium target

print("Loading Xenium target...")
adata_target = sc.read_h5ad(XENIUM_TARGET_H5AD)

adata_target.obs_names = adata_target.obs_names.astype(str)
adata_target.var_names = adata_target.var_names.astype(str)

print("Xenium target AnnData:")
print(adata_target)

if "spatial" not in adata_target.obsm:
    raise ValueError("Xenium target is missing obsm['spatial']; VISTA needs target spatial coordinates.")

print("Target spatial shape:", adata_target.obsm["spatial"].shape)


In [ ]:
# Prepare VISTA-required obs fields

if "names" not in adata_ref.obs.columns:
    adata_ref.obs["names"] = adata_ref.obs_names.astype(str)

if "names" not in adata_target.obs.columns:
    adata_target.obs["names"] = adata_target.obs_names.astype(str)

if "batch" not in adata_target.obs.columns:
    adata_target.obs["batch"] = "Br8667_xenium"

# Reference batch is optional, but useful if there is a sample column.
# If you know the exact metadata column for the 10 Huuki-Myers samples, set it here.
REFERENCE_BATCH_KEY = None

for col in ["sample_id", "sample", "SampleID", "Sample", "donor_id", "donor", "subject_id", "subject", "BrNum", "brnum"]:
    if col in adata_ref.obs.columns:
        REFERENCE_BATCH_KEY = col
        break

print("Reference batch key detected:", REFERENCE_BATCH_KEY)
print("Target batch key: batch")


In [ ]:
# Select common genes

common_genes = sorted(set(adata_ref.var_names) & set(adata_target.var_names))
print("Common genes between Huuki snRNA reference and Xenium target:", len(common_genes))

if len(common_genes) < 10:
    raise ValueError(
        f"Only {len(common_genes)} common genes found. "
        "This may mean one file uses Ensembl IDs and the other uses gene symbols."
    )

if RUN_MODE == "smoke":
    selected_genes = common_genes[: min(SMOKE_GENES, len(common_genes))]
else:
    selected_genes = common_genes

print("Selected genes:", len(selected_genes))
print(selected_genes[:30])


In [ ]:
# Create one reference-target dataset for this run

rng = np.random.default_rng(SEED)

if RUN_MODE == "smoke":
    ref_n = min(SMOKE_REF_CELLS, adata_ref.n_obs)
    target_n = min(SMOKE_TARGET_CELLS, adata_target.n_obs)
    max_epochs = SMOKE_EPOCHS

    ref_idx = rng.choice(adata_ref.n_obs, size=ref_n, replace=False)
    target_idx = rng.choice(adata_target.n_obs, size=target_n, replace=False)

    ref_run = adata_ref[ref_idx, selected_genes].copy()
    target_run = adata_target[target_idx, selected_genes].copy()
else:
    max_epochs = FULL_EPOCHS
    ref_run = adata_ref[:, selected_genes].copy()
    target_run = adata_target[:, selected_genes].copy()

# Make sure VISTA fields survive slicing
ref_run.obs["names"] = ref_run.obs_names.astype(str)
target_run.obs["names"] = target_run.obs_names.astype(str)
target_run.obs["batch"] = target_run.obs["batch"].astype(str)

neighbor_size = min(NEIGHBOR_SIZE, max(1, target_run.n_obs - 1))

print("Run mode:", RUN_MODE)
print("Reference run shape:", ref_run.shape)
print("Target run shape:", target_run.shape)
print("Max epochs:", max_epochs)
print("Neighbor size:", neighbor_size)


In [ ]:
# Train VISTA and impute

start = time.time()

print("Setting up target AnnData...")
GIMVI_GCN.setup_anndata(
    target_run,
    batch_key="batch",
    obs_names="names",
)

print("Setting up reference AnnData...")
if REFERENCE_BATCH_KEY is not None and REFERENCE_BATCH_KEY in ref_run.obs.columns:
    # Use sample/donor batch if available.
    GIMVI_GCN.setup_anndata(
        ref_run,
        batch_key=REFERENCE_BATCH_KEY,
        obs_names="names",
    )
else:
    GIMVI_GCN.setup_anndata(ref_run)

print("Initializing VISTA model...")
model = GIMVI_GCN(
    ref_run,
    target_run,
    n_latent=N_LATENT,
    neighbor_size=neighbor_size,
)

print("Training VISTA...")
model.train(
    max_epochs=max_epochs,
    train_size=1.0,
    validation_size=0.0,
    use_gpu="cuda:0" if torch.cuda.is_available() else False,
    batch_size=BATCH_SIZE,
)

print("Getting imputed values...")
pred = model.get_imputed_values(
    normalized=False,
    batch_size=BATCH_SIZE,
)[0]

elapsed = time.time() - start

print("Prediction shape:", pred.shape)
print(f"Finished VISTA one-run test in {elapsed:.2f} seconds")


In [ ]:
# Save outputs

summary = pd.DataFrame([{
    "run_mode": RUN_MODE,
    "reference": str(HUUKI_COUNTS_CSV),
    "reference_metadata": str(HUUKI_METADATA_CSV),
    "target": str(XENIUM_TARGET_H5AD),
    "reference_cells": ref_run.n_obs,
    "target_cells": target_run.n_obs,
    "genes": ref_run.n_vars,
    "epochs": max_epochs,
    "neighbor_size": neighbor_size,
    "prediction_shape": str(pred.shape),
    "seconds": elapsed,
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "",
}])

summary_path = OUT_DIR / f"vista_huuki_snrna_reference_xenium_target_{RUN_MODE}_summary.csv"
summary.to_csv(summary_path, index=False)

# Save only smoke predictions as CSV to avoid giant files in full mode.
if RUN_MODE == "smoke":
    pred_df = pd.DataFrame(
        pred,
        index=target_run.obs_names,
        columns=ref_run.var_names,
    )
    pred_path = OUT_DIR / "vista_huuki_snrna_reference_xenium_target_smoke_predictions.csv"
    pred_df.to_csv(pred_path)
    print("Saved prediction CSV:", pred_path)

print("Saved summary:", summary_path)
print(summary)


## Advisor wording

> I ran one VISTA feasibility test where the reference is the Huuki-Myers snRNA-seq count matrix plus metadata, and the target is Br8667 Xenium. The Huuki-Myers reference contains multiple biological samples, but I am not running 10 repeated tests. This is only a testing-GPU feasibility run to confirm that VISTA can initialize, train, and produce imputed values for the Huuki snRNA → Xenium setup.
